# Assignment 4: Unsupervised Learning; Clustering and Recommendation

Group 70: Michael Massaad (300293612) & Gabriel Zohrob (300309391)

Work Split:
- Gabriel Zohrob: Worked on data preparation and implemented Study 1 and 2

- Michael Massaad: Worked on data preparation and implemented Study 3 and 4

## Description of The Movies Dataset

Link: https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset

Author: Rounak Banik

Purpose: This dataset was originally assembled as part of a data science capstone project with the goal of performing extensive exploratory data analysis on movie data to better understand trends in the film industry. It was also designed to support the development of different types of recommendation systems by combining movie metadata with user rating data from the MovieLens dataset.

The dataset includes two main components:
- movie metadata, such as title, genres, overview, popularity, runtime, budget, revenue, and rating statistics (Study 1, 2, 3)
- user rating data, which captures how individual users rate different movies (Study 4)

Shape:
- movies_metadata.csv: 45,466 rows × 24 columns
- ratings_small.csv: 100,004 rows × 4 columns

Features: 

For the movies_metadata.csv:
- adult: Indicates whether the movie is adult content (TRUE/FALSE) – Categorical
- belongs_to_collection: Collection or franchise the movie belongs to (JSON structure) – Categorical
- budget: Production budget of the movie in USD – Numerical
- genres: List of genres associated with the movie (e.g., Action, Comedy) – Categorical
- homepage: Official website URL of the movie – Categorical
- id: Unique identifier for each movie – Numerical
- imdb_id: IMDb identifier for the movie – Categorical
- original_language: Original language of the movie (e.g., en, fr) – Categorical
- original_title: Original title of the movie – Categorical
- overview: Summary or description of the movie’s plot – Categorical
- popularity: Popularity score assigned to the movie – Numerical
- poster_path: Path to the movie’s poster image – Categorical
- production_companies: Companies involved in producing the movie – Categorical
- production_countries: Countries where the movie was produced – Categorical
- release_date: Date the movie was released – Categorical
- revenue: Revenue generated by the movie in USD – Numerical
- runtime: Duration of the movie in minutes – Numerical
- spoken_languages: Languages spoken in the movie – Categorical
- status: Release status of the movie (e.g., Released, Rumored) – Categorical
- tagline: Promotional tagline or slogan of the movie – Categorical
- title: Title of the movie – Categorical
- video: Indicates if a video is associated with the movie (TRUE/FALSE) – Categorical
- vote_average: Average rating given to the movie – Numerical
- vote_count: Total number of user votes – Numerical

For the ratings_small.csv: 
- userId: Unique identifier for each user – Numerical
- movieId: Unique identifier for each movie (links to movies dataset) – Numerical
- rating: Rating given by the user to the movie (typically from 0.5 to 5.0) – Numerical
- timestamp: Time at which the rating was given (Unix timestamp format) – Numerical

Explanation: 
This dataset was chosen because it provides both movie metadata and user ratings, allowing for a wide range of data analysis tasks. It enables the exploration of relationships between movie attributes (such as budget, popularity, and genres) and user behavior (ratings). Additionally, it supports tasks such as predicting movie success and building recommendation systems, making it well-suited for this assignment.


## Data Cleaning

In [32]:
# ===============================
# Imports
# ===============================
import pandas as pd
import numpy as np
import ast

# Optional display settings for easier notebook viewing
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

In [33]:
# ===============================
# Load datasets
# ===============================
movies_url = "https://raw.githubusercontent.com/gabrielzohrob/CSI-4142---Assignment-4/refs/heads/main/movies_metadata.csv"
ratings_url = "https://raw.githubusercontent.com/gabrielzohrob/CSI-4142---Assignment-4/refs/heads/main/ratings_small.csv"

movies = pd.read_csv(movies_url, low_memory=False)
ratings = pd.read_csv(ratings_url)

print("movies_metadata shape:", movies.shape)
print("ratings_small shape:", ratings.shape)

movies_metadata shape: (45466, 24)
ratings_small shape: (100004, 4)


In [34]:
# ===============================
# Basic inspection
# ===============================

print("Movies columns:")
print(movies.columns.tolist())

print("\nRatings columns:")
print(ratings.columns.tolist())

print("\nMovies preview:")
display(movies.head())

print("\nRatings preview:")
display(ratings.head())

Movies columns:
['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']

Ratings columns:
['userId', 'movieId', 'rating', 'timestamp']

Movies preview:


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', 'poster_path': '/7G9915LfUQ2lVfwMEEhDsn3kT4B.jpg', 'backdrop_path': '/9FBwqcd9IRruEDUrTdcaafOMKUq.jpg'}",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circum...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,"[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}, {'id': 10751, 'name': 'Family'}]",NaN,8844,tt0113497,en,Jumanji,"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- in...",17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[{'name': 'TriStar Pictures', 'id': 559}, {'name': 'Teitler Film', 'id': 2550}, {'name': 'Interscope Communications', 'id': 10201}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': 'fr', 'name': 'Français'}]",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collection', 'poster_path': '/nLvUdqgPgm3F85NMCii9gVFUcet.jpg', 'backdrop_path': '/hypTnLot2z8wpFS7qwsQHW1uV8u.jpg'}",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, 'name': 'Comedy'}]",NaN,15602,tt0113228,en,Grumpier Old Men,"A family wedding reignites the ancient feud between next-door neighbors and fishing buddies John and Max. Meanwhile, a sultry Italian divorcée opens a restaurant at the local bait shop, alarming t...",11.7129,/6ksm1sjKMFLbO7UY2i6G1ju9SML.jpg,"[{'name': 'Warner Bros.', 'id': 6194}, {'name': 'Lancaster Gate', 'id': 19464}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for Love.,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'name': 'Drama'}, {'id': 10749, 'name': 'Romance'}]",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the women are holding their breath, waiting for the elusive ""good man"" to break a string of less-than-stellar lovers. Friends and confidants Vannah, Bernie, ...",3.859495,/16XOMpEaLWkrcPqSQqhTmeJuqQl.jpg,"[{'name': 'Twentieth Century Fox Film Corporation', 'id': 306}]","[{'iso_3166_1': 'US', 'name': 'United States of America'}]",1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself... and never let you forget it.,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Collection', 'poster_path': '/nts4iOmNnq7GNicycMJ9pSAn204.jpg', 'backdrop_path': '/7qwE57OVZmMJChBpLEbJEmzUydk.jpg'}",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,"Just when George Banks has recovered from his daughter's wedding, he receives the news that she's pregnant ... and that George's wife, Nina, is expecting too. He was planning on selling their home...",8.387519,/e64sOI48hQXyru7naBFyssKFxVd.jpg,"[{'name': 'Sandollar Productions', 'id': 5842}, {'name': 'Touchstone Pictures', 'id': 9195}]","[{'iso_3166_1': '


Ratings preview:


,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


In [35]:
# ===============================
# Inspect data types
# ===============================
dtypes_table = pd.DataFrame({
    "Column": movies.columns,
    "Data Type": movies.dtypes.values
})

dtypes_table

,Column,Data Type
0,adult,str
1,belongs_to_collection,str
2,budget,str
3,genres,str
4,homepage,str
5,id,str
6,imdb_id,str
7,original_language,str
8,original_title,str
9,overview,str


In [36]:
# ===============================
# Check missing values
# ===============================
missing_table = pd.DataFrame({
    "Column": movies.columns,
    "Missing Values": movies.isnull().sum().values
})

missing_table = missing_table.sort_values(by="Missing Values", ascending=False)

missing_table

,Column,Missing Values
1,belongs_to_collection,40972
4,homepage,37684
19,tagline,25054
9,overview,954
11,poster_path,386
16,runtime,263
18,status,87
14,release_date,87
6,imdb_id,17
7,original_language,11


In [37]:
# ===============================
# Check problematic 'id' column
# ===============================
# Try converting id to numeric to see errors
invalid_ids = pd.to_numeric(movies['id'], errors='coerce').isnull().sum()

id_check_table = pd.DataFrame({
    "Check": ["Invalid ID values (non-numeric)"],
    "Count": [invalid_ids]
})

id_check_table

,Check,Count
0,Invalid ID values (non-numeric),3


In [38]:
# ===============================
# Fix 'id' column (CRITICAL STEP)
# ===============================

# Convert id to numeric (invalid values → NaN)
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')

# Count how many invalid IDs we have
invalid_ids = movies['id'].isnull().sum()
print("Invalid ID rows:", invalid_ids)

# Drop rows with invalid IDs
movies = movies.dropna(subset=['id'])

# Convert to integer
movies['id'] = movies['id'].astype(int)

# Verify
print("New movies shape:", movies.shape)

Invalid ID rows: 3
New movies shape: (45463, 24)


The `id` column was cleaned to ensure it can be reliably used as a unique identifier for each movie. Since some entries contained non-numeric values, the column was converted to a numeric format and invalid rows were removed. A total of **3 rows with invalid IDs** were identified and dropped, resulting in a cleaned dataset with **45,463 rows and 24 columns**. This step is essential for correctly linking the movies dataset with the ratings dataset in later stages of the analysis.
